# Feast offline-to-online training and inference

This notebook follows the [Feast quickstart](../../develop/components/feast/quickstart.mdx) and runs a small, CPU-only demonstration:

1. create a Parquet batch dataset;
2. define and register Feast entities, feature views, and a feature service;
3. retrieve historical features and train a sample NumPy model;
4. materialize the same features into the Feast online store; and
5. deploy a KServe `InferenceService` whose model server reads the online features before predicting.

The notebook assumes `kubectl`, Feast 0.61.x, NumPy, pandas, PyArrow, `requests`, and KServe are available. It uses a PVC for the model artifact and the published Feast feature-server image for the custom serving runtime. The same flow is automated by [`e2e/cases/c16_feast_offline_online.sh`](../../../e2e/cases/c16_feast_offline_online.sh).

In [ ]:
import base64
import json
import os
import subprocess
import textwrap
import time
from pathlib import Path

import numpy as np
import pandas as pd
import requests

NAMESPACE = os.environ.get("FEAST_NAMESPACE", "feast-demo")
FEATURESTORE_NAME = os.environ.get("FEAST_FEATURESTORE", "feast-notebook")
FEAST_PROJECT = os.environ.get("FEAST_PROJECT", "feast_demo")
MODEL_PVC = os.environ.get("FEAST_MODEL_PVC", "feast-notebook-model")
MODEL_RUNTIME = os.environ.get("FEAST_MODEL_RUNTIME", "feast-numpy-runtime")
MODEL_NAME = os.environ.get("FEAST_MODEL_NAME", "feast-online-model")
MODEL_IMAGE = os.environ.get("FEAST_MODEL_IMAGE", "build-harbor.alauda.cn/mlops/feast/feature-server:0.61.0")
REPO = Path("feast-notebook-repo")
DATA_DIR = REPO / "data"
MODEL_DIR = Path("feast-notebook-model")
REPO.mkdir(exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(exist_ok=True)

def kubectl(*args, input_text=None, check=True):
    result = subprocess.run(["kubectl", *args], input=input_text, text=True, capture_output=True)
    if check and result.returncode:
        raise RuntimeError(f"kubectl {' '.join(args)} failed: {result.stderr}")
    return result.stdout.strip()

print({"namespace": NAMESPACE, "featurestore": FEATURESTORE_NAME, "model_pvc": MODEL_PVC})

## 1. Prepare the FeatureStore operand

The quickstart requires `registry.local.server: {}` when a client outside the Feast pod runs `feast apply`. The UI is optional for this workflow, but is enabled here for inspection. If you already have a suitable `FeatureStore`, set `FEAST_FEATURESTORE` and skip this cell.

In [ ]:
for namespace in (NAMESPACE, "feast-operator-system"):
    namespace_yaml = kubectl("create", "namespace", namespace, "--dry-run=client", "-o", "yaml")
    kubectl("apply", "-f", "-", input_text=namespace_yaml)
featurestore_yaml = f"""
apiVersion: feast.dev/v1
kind: FeatureStore
metadata:
  name: {FEATURESTORE_NAME}
  namespace: {NAMESPACE}
spec:
  feastProject: {FEAST_PROJECT}
  services:
    registry:
      local:
        server: {{}}
    ui: {{}}
"""
kubectl("apply", "-f", "-", input_text=featurestore_yaml)
deadline = time.time() + 600
while time.time() < deadline:
    phase = kubectl("get", "featurestore", FEATURESTORE_NAME, "-n", NAMESPACE, "-o", "jsonpath={.status.phase}", check=False)
    print(phase or "Pending")
    if phase == "Ready":
        break
    if phase == "Failed":
        raise RuntimeError(kubectl("describe", "featurestore", FEATURESTORE_NAME, "-n", NAMESPACE, check=False))
    time.sleep(10)
else:
    raise TimeoutError("FeatureStore did not become Ready")

client_config_map = kubectl("get", "featurestore", FEATURESTORE_NAME, "-n", NAMESPACE, "-o", "jsonpath={.status.clientConfigMap}")
client_config = kubectl("get", "configmap", client_config_map, "-n", NAMESPACE, "-o", r"jsonpath={.data.feature_store\.yaml}")
print("FeatureStore is Ready; client config length:", len(client_config))

## 2. Prepare a Parquet batch source

The event timestamp is required by Feast for point-in-time historical retrieval. The label is kept in the entity dataframe for model training and is not registered as a feature.

In [ ]:
rng = np.random.default_rng(7)
n_rows = 240
events = pd.DataFrame({
    "driver_id": (np.arange(n_rows) % 12 + 1).astype("int64"),
    "event_timestamp": pd.date_range("2026-01-01", periods=n_rows, freq="h", tz="UTC"),
})
events["created"] = events["event_timestamp"] + pd.to_timedelta(1, unit="m")
events["conv_rate"] = (0.25 + 0.55 * rng.random(n_rows)).astype("float32")
events["acc_rate"] = (0.50 + 0.45 * rng.random(n_rows)).astype("float32")
events["avg_daily_trips"] = rng.integers(2, 20, size=n_rows).astype("int64")
events["label"] = ((events["conv_rate"] * 2 + events["acc_rate"] + events["avg_daily_trips"] / 20) > 1.8).astype("int64")
parquet_path = DATA_DIR / "driver_stats.parquet"
events.to_parquet(parquet_path, index=False)
print(parquet_path, events.shape)
events.head()

## 3. Define and register offline features

This follows the quickstart’s `Entity` + `FileSource` + `FeatureView` + `FeatureService` pattern. For the operand’s default PVC-backed SQLite online store, the cell also runs `feast apply` inside the online-store pod once, creating the feature-view table before remote materialization writes to it.

In [ ]:
(REPO / "features.py").write_text(textwrap.dedent("""
    from datetime import timedelta
    from feast import Entity, FeatureService, FeatureView, Field, FileSource
    from feast.data_format import ParquetFormat
    from feast.types import Float32, Int64
    from feast.value_type import ValueType

    driver = Entity(name="driver", join_keys=["driver_id"], value_type=ValueType.INT64)
    driver_stats_source = FileSource(
        name="driver_stats_source", path="data/driver_stats.parquet",
        file_format=ParquetFormat(), timestamp_field="event_timestamp",
        created_timestamp_column="created",
    )
    driver_hourly_stats = FeatureView(
        name="driver_hourly_stats", entities=[driver], ttl=timedelta(days=365),
        schema=[
            Field(name="conv_rate", dtype=Float32),
            Field(name="acc_rate", dtype=Float32),
            Field(name="avg_daily_trips", dtype=Int64),
        ], online=True, source=driver_stats_source,
    )
    driver_activity_v1 = FeatureService(name="driver_activity_v1", features=[driver_hourly_stats])
"""))

# Copy the platform-generated config and make the online/registry certificates
# available to this notebook process. The serving pod will mount the original
# /tls paths; keep that copy separately for the model artifact.
runtime_config = client_config
local_config = runtime_config
for secret_name, original_path, local_name in [
    (f"feast-{FEATURESTORE_NAME}-online-tls", "/tls/online/tls.crt", "online-tls.crt"),
    (f"feast-{FEATURESTORE_NAME}-registry-tls", "/tls/registry/tls.crt", "registry-tls.crt"),
]:
    cert_b64 = kubectl("get", "secret", secret_name, "-n", NAMESPACE, "-o", r"jsonpath={.data.tls\.crt}", check=False)
    if cert_b64:
        cert_path = (REPO / local_name).resolve()
        cert_path.write_bytes(base64.b64decode(cert_b64))
        local_config = local_config.replace(original_path, str(cert_path))
(REPO / "feature_store.yaml").write_text(local_config)
(MODEL_DIR / "feature_store.yaml").write_text(runtime_config)
(MODEL_DIR / "features.py").write_text((REPO / "features.py").read_text())
subprocess.run(["feast", "--chdir", str(REPO), "apply"], check=True)

# Feast 0.61's remote online-store client does not create SQLite tables.
# Apply once in the operand's local repository to create that infrastructure.
online_pod = kubectl("get", "pods", "-n", NAMESPACE, "-l", f"feast.dev/name={FEATURESTORE_NAME}",
                     "-o", "jsonpath={.items[0].metadata.name}")
if not online_pod:
    raise RuntimeError("FeatureStore online pod was not found")
kubectl("cp", str(REPO / "features.py"),
        f"{NAMESPACE}/{online_pod}:/feast-data/{FEAST_PROJECT}/feature_repo/feature_definitions.py",
        "-c", "online")
kubectl("exec", "-n", NAMESPACE, online_pod, "-c", "online", "--", "bash", "-c",
        f"cd /feast-data/{FEAST_PROJECT}/feature_repo && feast apply")
print("Feature definitions registered; online-store infrastructure created")

## 4. Retrieve offline features and train a sample model

The model is deliberately a small NumPy linear classifier so the example does not require a second private training image. The training matrix comes from Feast’s historical feature retrieval, not directly from the raw Parquet columns.

In [ ]:
from feast import FeatureStore

store = FeatureStore(repo_path=str(REPO))
entity_df = events[["driver_id", "event_timestamp", "label"]].copy()
training_df = store.get_historical_features(
    entity_df=entity_df,
    features=[
        "driver_hourly_stats:conv_rate",
        "driver_hourly_stats:acc_rate",
        "driver_hourly_stats:avg_daily_trips",
    ],
).to_df().dropna()

feature_columns = ["conv_rate", "acc_rate", "avg_daily_trips"]
X = training_df[feature_columns].to_numpy(dtype="float64")
y = training_df["label"].to_numpy(dtype="float64")
X_bias = np.column_stack([np.ones(len(X)), X])
weights = np.linalg.pinv(X_bias) @ y
np.savez(MODEL_DIR / "model.npz", weights=weights, feature_columns=np.array(feature_columns))
print("historical rows:", len(training_df), "weights:", weights)

## 5. Materialize and verify online features

`materialize_incremental` copies the registered batch features into the online store. The inference server below uses the same feature service and entity key at request time.

In [ ]:
end_date = events["event_timestamp"].max().to_pydatetime() + pd.Timedelta(hours=1)
store.materialize_incremental(end_date)
online = store.get_online_features(
    features=store.get_feature_service("driver_activity_v1"),
    entity_rows=[{"driver_id": 1}, {"driver_id": 2}],
).to_df()
online

## 6. Define the online model server

The server loads the trained NumPy weights from the model PVC, queries Feast online features, and returns a KServe v2 response. The Feast online and registry TLS secrets are mounted by the `ServingRuntime`.

In [ ]:
server_source = textwrap.dedent("""
    import os
    import numpy as np
    from fastapi import Body, FastAPI
    from feast import FeatureStore
    import uvicorn

    MODEL_NAME = os.getenv("MODEL_NAME", "feast-online-model")
    weights = np.load("/mnt/models/model.npz")["weights"]
    store = FeatureStore(repo_path="/mnt/models")
    feature_service = store.get_feature_service("driver_activity_v1")
    app = FastAPI()

    @app.get("/v2/health/live")
    @app.get("/v2/health/ready")
    def ready():
        return {"ready": True}

    @app.get("/v2/models/{model_name}")
    @app.get("/v2/models/{model_name}/ready")
    def model_ready(model_name: str):
        return {"name": model_name, "ready": model_name == MODEL_NAME}

    @app.post("/v2/models/{model_name}/infer")
    def infer(model_name: str, payload: dict = Body(...)):
        ids = next(item for item in payload["inputs"] if item["name"] == "driver_id")["data"]
        rows = [{"driver_id": int(driver_id)} for driver_id in ids]
        values = store.get_online_features(features=feature_service, entity_rows=rows).to_dict()
        def column(name):
            if name in values:
                return values[name]
            return values[next(key for key in values if key.endswith("__" + name))]
        X = np.column_stack([np.ones(len(ids)), column("conv_rate"), column("acc_rate"), column("avg_daily_trips")])
        prediction = (X @ weights).astype("float32")
        return {"model_name": model_name, "outputs": [{"name": "prediction", "shape": [len(ids)], "datatype": "FP32", "data": prediction.tolist()}]}

    if __name__ == "__main__":
        uvicorn.run(app, host="0.0.0.0", port=8080)
""")
(MODEL_DIR / "server.py").write_text(server_source)
print(MODEL_DIR / "server.py")

## 7. Stage the model and start KServe

Create a model PVC before running this section. The temporary stager pod copies the local model artifact into it; KServe then consumes it through `storageUri: pvc://...`. The runtime mounts the Feast online and registry certificates at the paths referenced by the generated client configuration. The `RawDeployment` annotation allows a direct predictor Service where the cluster permits it; the final cell verifies the predictor Deployment itself so it also works on clusters whose KServe policy selects Standard mode.

In [ ]:
pvc_yaml = f"""
apiVersion: v1
kind: PersistentVolumeClaim
metadata:
  name: {MODEL_PVC}
  namespace: {NAMESPACE}
spec:
  accessModes: [ReadWriteOnce]
  resources:
    requests:
      storage: 1Gi
"""
kubectl("apply", "-f", "-", input_text=pvc_yaml)
stager = f"""
apiVersion: v1
kind: Pod
metadata:
  name: feast-model-stager
  namespace: {NAMESPACE}
spec:
  restartPolicy: Never
  containers:
  - name: stager
    image: {MODEL_IMAGE}
    command: [bash, -c, sleep 3600]
    volumeMounts:
    - name: model
      mountPath: /mnt/models
  volumes:
  - name: model
    persistentVolumeClaim:
      claimName: {MODEL_PVC}
"""
kubectl("apply", "-f", "-", input_text=stager)
kubectl("wait", "--for=condition=Ready", "pod/feast-model-stager", "-n", NAMESPACE, "--timeout=180s")
kubectl("cp", str(MODEL_DIR / "model.npz"), f"{NAMESPACE}/feast-model-stager:/mnt/models/model.npz")
kubectl("cp", str(MODEL_DIR / "server.py"), f"{NAMESPACE}/feast-model-stager:/mnt/models/server.py")
kubectl("cp", str(MODEL_DIR / "features.py"), f"{NAMESPACE}/feast-model-stager:/mnt/models/features.py")
kubectl("cp", str(MODEL_DIR / "feature_store.yaml"), f"{NAMESPACE}/feast-model-stager:/mnt/models/feature_store.yaml")
kubectl("delete", "pod", "feast-model-stager", "-n", NAMESPACE, "--wait=true")

runtime_yaml = f"""
apiVersion: serving.kserve.io/v1alpha1
kind: ServingRuntime
metadata:
  name: {MODEL_RUNTIME}
  namespace: {NAMESPACE}
spec:
  containers:
  - name: kserve-container
    image: {MODEL_IMAGE}
    command: [python, /mnt/models/server.py]
    ports:
    - containerPort: 8080
      name: http1
      protocol: TCP
    env:
    - name: MODEL_NAME
      value: {MODEL_NAME}
    volumeMounts:
    - name: online-tls
      mountPath: /tls/online
      readOnly: true
    - name: registry-tls
      mountPath: /tls/registry
      readOnly: true
  protocolVersions: [v2]
  supportedModelFormats:
  - name: feast-numpy
    version: "1"
  volumes:
  - name: online-tls
    secret:
      secretName: feast-{FEATURESTORE_NAME}-online-tls
  - name: registry-tls
    secret:
      secretName: feast-{FEATURESTORE_NAME}-registry-tls
"""
isvc_yaml = f"""
apiVersion: serving.kserve.io/v1beta1
kind: InferenceService
metadata:
  name: {MODEL_NAME}
  namespace: {NAMESPACE}
  annotations:
    serving.kserve.io/deploymentMode: RawDeployment
spec:
  predictor:
    model:
      modelFormat:
        name: feast-numpy
        version: "1"
      protocolVersion: v2
      runtime: {MODEL_RUNTIME}
      storageUri: pvc://{MODEL_PVC}
      resources:
        requests:
          cpu: "100m"
          memory: 256Mi
        limits:
          cpu: "1"
          memory: 1Gi
"""
kubectl("apply", "-f", "-", input_text=runtime_yaml)
kubectl("apply", "-f", "-", input_text=isvc_yaml)
print(kubectl("get", "inferenceservice", MODEL_NAME, "-n", NAMESPACE))

## 8. Wait for the service and send an online-feature prediction

When the predictor Deployment has an available replica, send entity IDs to the KServe v2 endpoint. The server looks those IDs up in Feast’s online store and combines the returned features with the trained weights.

In [ ]:
deadline = time.time() + 900
predictor_deployment = f"{MODEL_NAME}-predictor"
while time.time() < deadline:
    deployment = json.loads(kubectl("get", "deployment", predictor_deployment, "-n", NAMESPACE, "-o", "json"))
    available = deployment.get("status", {}).get("availableReplicas", 0) or 0
    print({"availableReplicas": available})
    if available >= 1:
        break
    time.sleep(10)
else:
    raise TimeoutError("KServe predictor deployment did not become available")

isvc_status = json.loads(kubectl("get", "inferenceservice", MODEL_NAME, "-n", NAMESPACE, "-o", "json"))
print("Ingress URL (if configured):", isvc_status.get("status", {}).get("url"))

port_forward = subprocess.Popen(
    ["kubectl", "port-forward", f"service/{predictor_deployment}", "18080:80", "-n", NAMESPACE],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)
base_url = "http://127.0.0.1:18080"
try:
    deadline = time.time() + 60
    while time.time() < deadline:
        if port_forward.poll() is not None:
            raise RuntimeError("kubectl port-forward exited unexpectedly")
        try:
            if requests.get(f"{base_url}/v2/health/ready", timeout=2).ok:
                break
        except requests.RequestException:
            pass
        time.sleep(2)
    else:
        raise TimeoutError("KServe predictor endpoint did not become ready")

    response = requests.post(
        f"{base_url}/v2/models/{MODEL_NAME}/infer",
        json={"inputs": [{"name": "driver_id", "shape": [2], "datatype": "INT64", "data": [1, 2]}]},
        timeout=30,
    )
    response.raise_for_status()
    prediction = response.json()
    print(json.dumps(prediction, indent=2))
finally:
    port_forward.terminate()
    try:
        port_forward.wait(timeout=5)
    except subprocess.TimeoutExpired:
        port_forward.kill()
        port_forward.wait()